In [ ]:
# Run first: locate the repository root from any notebook subfolder.
from pathlib import Path
import os
import sys
project_root = next((candidate for candidate in [Path.cwd(), *Path.cwd().parents]
                     if (candidate / "app.py").is_file()
                     and (candidate / "retrieval_final.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Start Jupyter within the project repository.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
# Imports
import pandas as pd

from email import policy
from email.parser import Parser

In [ ]:
# Loads the original dataset
RAW_EMAILS_FILE = "emails.csv"

raw_email_rows = pd.read_csv(
    RAW_EMAILS_FILE
)

print(raw_email_rows.shape)
print(raw_email_rows.columns)
raw_email_rows.head()

In [ ]:
# initial dataset checks
print("Shape:", raw_email_rows.shape)

print("\nMissing values:")
print(raw_email_rows.isna().sum())

print("\nExact duplicate rows:")
print(raw_email_rows.duplicated().sum())

print("\nExample raw message:")
print(raw_email_rows.iloc[0]["message"][:1500])

In [ ]:
# Converts the raw emails into structured data
def parse_email(raw_message):

    msg = Parser(policy=policy.compat32).parsestr(raw_message)

    body = msg.get_payload()

    if isinstance(body, list):
        body = "\n".join(
            part.get_payload()
            for part in body
            if isinstance(part.get_payload(), str)
        )

    return {
        "email_id": msg.get("Message-ID", ""),
        "date": msg.get("Date", ""),
        "sender": msg.get("From", ""),
        "recipients": msg.get("To", ""),
        "subject": msg.get("Subject", ""),
        "body": body if isinstance(body, str) else ""
    }

In [ ]:
# Apply parser all emails
parsed_email_records = [
    parse_email(message)
    for message in raw_email_rows["message"]
]

parsed_email_fields = pd.DataFrame(parsed_email_records)

parsed_emails = pd.concat(
    [
        raw_email_rows[["file"]].reset_index(drop=True),
        parsed_email_fields
    ],
    axis=1
)

In [ ]:
print(parsed_emails.shape)

print("\nMissing values:")
print(parsed_emails.isna().sum())

parsed_emails[[
    "email_id",
    "date",
    "sender",
    "recipients",
    "subject",
    "body"
]].head()

In [ ]:
# Basic body inspection

parsed_emails["body"] = parsed_emails["body"].fillna("").astype(str)

body_character_counts = parsed_emails["body"].str.strip().str.len()

print("Total parsed emails:", len(parsed_emails))
print("Empty bodies:", (body_character_counts == 0).sum())
print("Bodies under 10 characters:", (body_character_counts < 10).sum())
print("Bodies under 20 characters:", (body_character_counts < 20).sum())
print("Bodies under 50 characters:", (body_character_counts < 50).sum())
print("Bodies under 100 characters:", (body_character_counts < 100).sum())

print("\nBody length statistics:")
print(body_character_counts.describe())

In [ ]:
body_character_counts = parsed_emails["body"].str.len()

for limit in [5000, 10000, 20000, 50000, 100000, 500000, 1000000]:
    print(
        f"Over {limit:,} characters:",
        (body_character_counts > limit).sum()
    )

In [ ]:
# the longest 30 emails
parsed_emails.assign(
    body_length=body_character_counts
).sort_values(
    "body_length",
    ascending=False
)[
    ["file", "subject", "body_length"]
].head(30)

In [ ]:
# prints teh first 1000 characters of the sample of long emails
for min_len, max_len in [
    (10000, 20000),
    (50000, 100000),
    (100000, 500000)
]:
    email_length_sample = parsed_emails[
        parsed_emails["body"].str.len().between(min_len, max_len)
    ].sample(3, random_state=42)

    for _, row in email_length_sample.iterrows():
        print("\n" + "=" * 80)
        print("SUBJECT:", row["subject"])
        print("LENGTH:", len(row["body"]))
        print(row["body"][:1000])

In [ ]:
# Reviewing duplicate emails
print("Total emails:", len(parsed_emails))
print("Duplicate message IDs:", parsed_emails["email_id"].duplicated().sum())
print("Unique message IDs:", parsed_emails["email_id"].nunique())

In [ ]:
print(
    "Duplicate subject + body:",
    parsed_emails.duplicated(subset=["subject", "body"]).sum()
)

In [ ]:
# Removes emails with same subject and body
cleaned_emails = parsed_emails.drop_duplicates(
    subset=["subject", "body"]
).reset_index(drop=True)

print("Original:", len(parsed_emails))
print("After removing duplicates:", len(cleaned_emails))

In [ ]:
# remove all large emails beyond 5000 characters
MAX_BODY_CHARACTERS = 50_000

emails_before_length_filter = len(cleaned_emails)
cleaned_emails = cleaned_emails[
    cleaned_emails["body"].fillna("").astype(str).str.len() <= MAX_BODY_CHARACTERS
].reset_index(drop=True)

print("Removed emails over 50,000 characters:", emails_before_length_filter - len(cleaned_emails))
print("Remaining cleaned emails:", len(cleaned_emails))

In [ ]:
# Save the cleaned dataset
cleaned_emails.to_csv("enron_clean.csv", index=False)

In [ ]:
# Create a subset for generating question answers for LoRa
enron_subset_8000 = cleaned_emails.sample(n=8_000, random_state=42).reset_index(drop=True)
enron_subset_8000.to_csv("enron_subset_8000.csv", index=False)
print(f"Saved {len(enron_subset_8000):,} emails to enron_subset_8000.csv")